# AI-Powered Data Science: Companion Notebook

Every code example from the book, in order, plus answers to the *Try It Yourself* exercises.

**Before you start:** upload `brewhaus_customers.csv` and `brewhaus_sales.csv` using the folder icon on the left, then choose **Runtime > Run all**.

## Chapter 2. Setting Up Your Toolkit

In [ ]:
print("Hello, data")

In [ ]:
import pandas as pd

customers = pd.read_csv("brewhaus_customers.csv")
customers.head()

**Try it yourself (answer):** load the sales file and show 10 rows.

In [ ]:
sales = pd.read_csv("brewhaus_sales.csv")
sales.head(10)

## Chapter 3. Prompting for Data Work

In [ ]:
customers.info()

In [ ]:
print(customers.columns.tolist())

In [ ]:
customers.drop(columns=["customer_id"]).head(3)

**Try it yourself (sample CRISP rewrites):**

1. *Find my best customers* → "I have a pandas DataFrame called customers (columns: ...). Act as a senior analyst. Rank customers by monthly_spend, then show the top 20 with their store, visits_90d and loyalty_member. Exclude the five office accounts spending over $500 and explain why. Comment each step."
2. *Fix the dates* → "signup_date and last_visit_date are text in mixed formats (2024-03-15, 03/15/2024, 15 Mar 2024). Dates are month-first. Convert both to datetime with pandas, turn unparseable values into missing, and report how many failed and the earliest and latest dates."
3. *Is the loyalty program working?* → "Compare loyalty members and non-members on average monthly_spend, visits_90d and churn rate in one grouped table. Then explain whether the differences could be caused by something other than the program itself."

## Chapter 4. Meet Your Data

In [ ]:
customers.shape

In [ ]:
customers.describe()

In [ ]:
customers["loyalty_member"].value_counts()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.hist(customers["monthly_spend"].dropna(), bins=40, range=(0, 100),
         color="gray", edgecolor="black")
plt.xlabel("Monthly spend ($)")
plt.ylabel("Number of customers")
plt.title("Most customers spend under $60 a month; a few spend far more")
plt.show()

In [ ]:
customers[customers["monthly_spend"] > 500]

In [ ]:
import seaborn as sns

corr = customers.corr(numeric_only=True)
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Greys")
plt.title("How BrewHaus numbers move together")
plt.show()

In [ ]:
customers[customers["monthly_spend"] < 500].corr(numeric_only=True)

In [ ]:
customers["age"].isna().mean()

**Try it yourself (answer):** the first five commands on the sales file.

In [ ]:
print(sales.shape)
sales.info()
sales.describe()

## Chapter 5. Cleaning Data

In [ ]:
raw = pd.read_csv("brewhaus_customers.csv")
customers = raw.copy()
print("Starting rows:", len(customers))

In [ ]:
print("Exact duplicate rows:", customers.duplicated().sum())

# Same ID, but rows that differ somewhere
dup_ids = customers["customer_id"].duplicated(keep=False)
print("Rows sharing an ID:", dup_ids.sum())

customers = customers.drop_duplicates()
print("Rows after removing exact duplicates:", len(customers))

In [ ]:
cleaned = (customers["avg_order_value"]
           .astype(str)
           .str.replace("$", "", regex=False)
           .str.strip())
customers["avg_order_value"] = pd.to_numeric(cleaned, errors="coerce")

print(customers["avg_order_value"].dtype)
print("Failed to convert:", customers["avg_order_value"].isna().sum())

In [ ]:
customers["city"].value_counts()

In [ ]:
print("Before:", customers["city"].nunique(), "distinct values")

customers["city"] = customers["city"].str.strip().str.title()
customers["city"] = customers["city"].replace({
    "Nyc": "New York",
    "New York City": "New York",
})

print("After:", customers["city"].nunique(), "distinct values")
print(customers["city"].value_counts())

In [ ]:
customers["loyalty_member"] = (customers["loyalty_member"]
                               .str.strip()
                               .str.lower()
                               .map({"yes": "Yes", "y": "Yes", "no": "No", "n": "No"}))
print(customers["loyalty_member"].value_counts(dropna=False))

In [ ]:
for col in ["signup_date", "last_visit_date"]:
    customers[col] = pd.to_datetime(customers[col], format="mixed", errors="coerce")
    print(col, "failed:", customers[col].isna().sum())

In [ ]:
print(customers["signup_date"].min(), customers["signup_date"].max())

In [ ]:
import numpy as np

bad_age = (customers["age"] < 16) | (customers["age"] > 100)
print("Impossible ages:", bad_age.sum())
customers.loc[bad_age, "age"] = np.nan

In [ ]:
customers.isna().sum()

In [ ]:
customers["age_missing"] = customers["age"].isna().astype(int)
customers["age"] = customers["age"].fillna(customers["age"].median())

In [ ]:
def clean_customers(df):
    """Clean the raw BrewHaus customer export and return a new DataFrame."""
    df = df.copy()
    start = len(df)

    df = df.drop_duplicates()

    aov = df["avg_order_value"].astype(str).str.replace("$", "", regex=False).str.strip()
    df["avg_order_value"] = pd.to_numeric(aov, errors="coerce")

    df["city"] = df["city"].str.strip().str.title().replace(
        {"Nyc": "New York", "New York City": "New York"})

    df["loyalty_member"] = (df["loyalty_member"].str.strip().str.lower()
                            .map({"yes": "Yes", "y": "Yes", "no": "No", "n": "No"}))

    for col in ["signup_date", "last_visit_date"]:
        df[col] = pd.to_datetime(df[col], format="mixed", errors="coerce")

    df.loc[(df["age"] < 16) | (df["age"] > 100), "age"] = np.nan
    df["age_missing"] = df["age"].isna().astype(int)
    df["age"] = df["age"].fillna(df["age"].median())

    print(f"Rows: {start} -> {len(df)}")
    return df

check = clean_customers(raw)
print("Matches step-by-step version:", check.equals(customers))
customers = check

In [ ]:
customers.to_csv("brewhaus_customers_clean.csv", index=False)

**Try it yourself (answer key):** the cleaned file has **4,950 rows** (5,000 minus 50 exact duplicates), `avg_order_value` as float64, exactly two loyalty values (plus a few missing), datetime date columns, and every age between 16 and 100.

## Chapter 6. Writing and Debugging Python

In [ ]:
store = "Downtown"         # text (a string)
visits = 42                # a whole number (an integer)
avg_order = 5.75           # a decimal (a float)
is_member = True           # True or False (a boolean)

stores = ["Downtown", "Riverside", "Airport", "University", "Mall"]
print(stores[0])     # Downtown: counting starts at 0
print(len(stores))   # 5

rent = {"Downtown": 8000, "Airport": 12000}
print(rent["Airport"])   # 12000

for s in stores:
    print("Checking store:", s)

if visits > 30:
    print("Frequent customer")
else:
    print("Occasional customer")

In [ ]:
def spend_per_visit(total_spend, visits):
    """Return average spend per visit."""
    if visits == 0:
        return 0
    return total_spend / visits

print(spend_per_visit(120, 24))   # 5.0

In [ ]:
loyal = customers[customers["loyalty_member"] == "Yes"]
big_loyal = customers[(customers["loyalty_member"] == "Yes") & (customers["monthly_spend"] > 100)]
customers.sort_values("monthly_spend", ascending=False).head(10)

In [ ]:
customers.groupby("store")["monthly_spend"].mean().sort_values(ascending=False)

In [ ]:
customers.groupby("store").agg(
    customers=("customer_id", "count"),
    avg_spend=("monthly_spend", "mean"),
    churn_rate=("churned", "mean"),
)

In [ ]:
store_info = pd.DataFrame({
    "store": ["Downtown", "Riverside", "Airport", "University", "Mall"],
    "region": ["Central", "East", "North", "West", "South"],
})
before = len(customers)
customers = customers.merge(store_info, on="store", how="left")
print(before, "->", len(customers))

**Try it yourself: three broken cells.** Run each, read the error, then fix it. The fixed versions follow.

In [ ]:
customers["Store"].value_counts()   # broken: KeyError

In [ ]:
customers["store"].value_counts()   # fixed: the column name is lowercase

In [ ]:
raw["avg_order_value"].mean()   # broken: TypeError, the column is text

In [ ]:
pd.to_numeric(raw["avg_order_value"].str.replace("$", "", regex=False), errors="coerce").mean()   # fixed

In [ ]:
# Broken: a silent bug. The lookup table lists Airport twice, so the merge copies rows.
targets = pd.DataFrame({"store": ["Downtown", "Airport", "Airport"], "target": [3000, 3500, 3600]})
merged = customers.merge(targets, on="store", how="left")
print(len(customers), "->", len(merged), "rows")   # row count went up!

In [ ]:
# Fixed: remove duplicate keys from the lookup table before merging.
targets = targets.drop_duplicates(subset="store", keep="last")
merged = customers.merge(targets, on="store", how="left")
print(len(customers), "->", len(merged), "rows")

## Chapter 7. Visualizations

In [ ]:
sales = pd.read_csv("brewhaus_sales.csv")
sales["date"] = pd.to_datetime(sales["date"])

In [ ]:
by_store = sales.groupby("store")["revenue"].sum().sort_values(ascending=False)
colors = ["black"] + ["lightgray"] * (len(by_store) - 1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(by_store.index, by_store.values, color=colors)
ax.set_title(f"{by_store.index[0]} leads all stores in total revenue")
ax.set_ylabel("Total revenue ($)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v/1e6:.1f}M"))
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
monthly = sales.groupby(sales["date"].dt.to_period("M"))["revenue"].sum()
monthly.index = monthly.index.to_timestamp()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(monthly.index, monthly.values, color="black", linewidth=2)
ax.set_title("Monthly revenue climbs each autumn and dips every summer")
ax.set_ylabel("Revenue ($)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
customers = pd.read_csv("brewhaus_customers_clean.csv")
# Leave out the five office accounts so they don't squash the chart
regular = customers[customers["monthly_spend"] < 500]

fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=regular, x="visits_90d", y="monthly_spend",
                hue="loyalty_member", style="loyalty_member",
                alpha=0.5, ax=ax)
ax.set_title("Frequent visitors spend more, and most of them "
             "are loyalty members")
ax.set_xlabel("Visits in the last 90 days")
ax.set_ylabel("Monthly spend ($)")
plt.show()

In [ ]:
fig.savefig("visits_vs_spend.png", dpi=200, bbox_inches="tight")

In [ ]:
import plotly.express as px

fig = px.line(monthly.reset_index(), x="date", y="revenue",
              title="Monthly revenue")
fig.show()

**Try it yourself (answer):** which store grew fastest from the first year to the second?

In [ ]:
sales["year"] = (sales["date"] >= "2025-07-01").map({False: "Year 1", True: "Year 2"})
yearly = sales.pivot_table(index="store", columns="year", values="revenue", aggfunc="sum")
growth = ((yearly["Year 2"] / yearly["Year 1"] - 1) * 100).sort_values(ascending=False)
print(growth.round(1))

colors = ["black"] + ["lightgray"] * (len(growth) - 1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(growth.index, growth.values, color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title(f"{growth.index[0]} grew fastest, up {growth.iloc[0]:.0f}% year over year")
ax.set_ylabel("Revenue growth (%)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## Chapter 8. Automating Reports

In [ ]:
import pandas as pd
from datetime import date

# ---- Settings ----
INPUT_FILE = "brewhaus_sales.csv"
DAYS = 7
OUTPUT_FILE = f"weekly_report_{date.today()}.xlsx"

# ---- Load ----
sales = pd.read_csv(INPUT_FILE)
sales["date"] = pd.to_datetime(sales["date"])

# ---- Define this week and last week ----
end = sales["date"].max()
this_week = sales[sales["date"] > end - pd.Timedelta(days=DAYS)]
last_week = sales[(sales["date"] <= end - pd.Timedelta(days=DAYS)) &
                  (sales["date"] > end - pd.Timedelta(days=2 * DAYS))]

# ---- Summarize ----
summary = this_week.groupby("store")[["revenue", "transactions"]].sum()
summary["revenue_last_week"] = last_week.groupby("store")["revenue"].sum()
summary["change_pct"] = ((summary["revenue"] / summary["revenue_last_week"] - 1) * 100).round(1)
summary = summary.sort_values("revenue", ascending=False)

# ---- Save ----
summary.to_excel(OUTPUT_FILE)
print(f"Saved {OUTPUT_FILE}")
print(summary)

In [ ]:
# The check: one store, by hand
mask = (sales["store"] == "Airport") & (sales["date"] > end - pd.Timedelta(days=DAYS))
print("Days in 'this week':", sales.loc[mask, "date"].nunique())
print("Airport revenue by hand:", round(sales.loc[mask, "revenue"].sum(), 2))

**LLM summary (optional).** Store your API key in Colab Secrets as `ANTHROPIC_API_KEY` first. This cell is skipped unless you run it yourself.

In [ ]:
!pip install anthropic

import anthropic
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

prompt = f"""You are a retail analyst. Write a 3-sentence summary of this week's
coffee shop results for store managers. Mention the top store and the biggest
change. Use only the numbers in this table; do not invent figures.

{summary.to_string()}"""

response = client.messages.create(
    model="claude-sonnet-5-5",   # check the provider's docs for current model names
    max_tokens=300,
    messages=[{"role": "user", "content": prompt}],
)
print(response.content[0].text)

In [ ]:
if (pd.Timestamp.today() - end).days > 3:
    print("WARNING: the sales file looks out of date. Newest date:", end.date())

**Try it yourself (answer):** a monthly report for the most recent full month.

In [ ]:
sales["month"] = sales["date"].dt.to_period("M")
last_day = sales["date"].max()
latest = last_day.to_period("M")
if last_day != last_day + pd.offsets.MonthEnd(0):
    latest -= 1                      # the newest month is incomplete
previous = latest - 1

this_month = sales[sales["month"] == latest].groupby("store")[["revenue", "transactions"]].sum()
this_month["revenue_prev_month"] = sales[sales["month"] == previous].groupby("store")["revenue"].sum()
this_month["change_pct"] = ((this_month["revenue"] / this_month["revenue_prev_month"] - 1) * 100).round(1)
this_month.to_excel(f"monthly_report_{latest}.xlsx")
print(latest, "vs", previous)
print(this_month.sort_values("revenue", ascending=False))

## Chapter 10. Your First Predictive Models

In [ ]:
import pandas as pd

df = pd.read_csv(
    "brewhaus_customers_clean.csv",
    parse_dates=["signup_date", "last_visit_date"],
)
df = df.dropna(subset=["avg_order_value",
                       "loyalty_member", "signup_date"])

snapshot = df["signup_date"].max()
df["tenure_days"] = (snapshot - df["signup_date"]).dt.days
print(len(df), "rows ready for modeling")

### Project A: regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

num_cols = ["age", "tenure_days"]
cat_cols = ["store", "loyalty_member", "favorite_drink"]

X = df[num_cols + cat_cols]      # features
y = df["avg_order_value"]         # target

# 1. Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# 2. Create: preparation + model in one pipeline
prep = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
reg = Pipeline([("prep", prep),
                ("model", LinearRegression())])

# 3. Fit
reg.fit(X_train, y_train)

# 4. Predict
preds = reg.predict(X_test)

# 5. Score
mae = mean_absolute_error(y_test, preds)
print(f"Average error: ${mae:.2f} per visit")

In [ ]:
baseline = [y_train.mean()] * len(y_test)
print("Baseline error:", mean_absolute_error(y_test, baseline))

# Keep Project A's answers under their own names for Chapter 11
y_test_reg, preds_reg = y_test, preds

### Project B: classification

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

num_cols = ["age", "age_missing", "visits_90d",
            "avg_order_value", "monthly_spend", "tenure_days"]
cat_cols = ["store", "loyalty_member", "favorite_drink"]

X = df[num_cols + cat_cols]
y = df["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

def make_prep():
    return ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])

logreg = Pipeline([
    ("prep", make_prep()),
    ("model", LogisticRegression(max_iter=1000)),
])
tree = Pipeline([
    ("prep", make_prep()),
    ("model", DecisionTreeClassifier(max_depth=4,
                                     random_state=42)),
])

logreg.fit(X_train, y_train)
tree.fit(X_train, y_train)

baseline = y_test.value_counts(normalize=True).max()
print(f"Baseline accuracy: {baseline:.1%}")
print(f"Logistic regression: {logreg.score(X_test, y_test):.1%}")
print(f"Decision tree: {tree.score(X_test, y_test):.1%}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

names = tree.named_steps["prep"].get_feature_names_out()
plt.figure(figsize=(14, 6))
plot_tree(tree.named_steps["model"], feature_names=names,
          class_names=["Stayed", "Churned"], filled=True,
          max_depth=2, fontsize=9)
plt.show()

In [ ]:
df["churn_risk"] = logreg.predict_proba(X)[:, 1]
at_risk = df.sort_values("churn_risk", ascending=False)
print(at_risk[["customer_id", "store", "churn_risk"]].head(10))

**Try it yourself (answer):** add `spend_per_visit` and retrain.

In [ ]:
df["spend_per_visit"] = (df["monthly_spend"] / df["visits_90d"].replace(0, np.nan)).fillna(0)
num_plus = num_cols + ["spend_per_visit"]
X2 = df[num_plus + cat_cols]
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.2, random_state=42, stratify=y)
prep2 = ColumnTransformer([
    ("num", StandardScaler(), num_plus),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
logreg2 = Pipeline([("prep", prep2), ("model", LogisticRegression(max_iter=1000))])
logreg2.fit(X2_train, y2_train)
print(f"Before: {logreg.score(X_test, y_test):.1%}  After: {logreg2.score(X2_test, y2_test):.1%}")

## Chapter 11. Evaluating, Improving and Explaining

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_estimator(
    logreg, X_test, y_test,
    display_labels=["Stayed", "Churned"], cmap="Greys")
plt.title("Churn model: where it's right and wrong")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test, logreg.predict(X_test),
    target_names=["Stayed", "Churned"]))

In [ ]:
from sklearn.metrics import r2_score

print("R-squared:", round(r2_score(y_test_reg, preds_reg), 2))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = Pipeline([
    ("prep", make_prep()),
    ("model", RandomForestClassifier(
        n_estimators=300, max_depth=8,
        class_weight="balanced", random_state=42)),
])
forest.fit(X_train, y_train)
print(classification_report(
    y_test, forest.predict(X_test),
    target_names=["Stayed", "Churned"]))

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(forest, X, y, cv=5, scoring="recall")
print("Recall per fold:", scores.round(2))
print("Average recall:", scores.mean().round(2))

In [ ]:
names = forest.named_steps["prep"].get_feature_names_out()
importance = pd.Series(
    forest.named_steps["model"].feature_importances_,
    index=names).sort_values()

importance.tail(10).plot(kind="barh", color="gray",
                         figsize=(7, 4))
plt.title("Recent visits matter most for predicting churn")
plt.xlabel("Importance")
plt.show()

**Try it yourself (answer):** compare recall for churners across models.

In [ ]:
from sklearn.metrics import recall_score

for name, model in [("Logistic regression", logreg), ("Decision tree", tree), ("Random forest (balanced)", forest)]:
    print(f"{name}: churner recall {recall_score(y_test, model.predict(X_test)):.0%}")

## Chapter 12. Capstone starter

In [ ]:
import seaborn as sns
titanic = sns.load_dataset("titanic")
titanic.head()

The Titanic dataset downloads from the internet the first time. From here, follow the ten-prompt script in Chapter 12.